# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [ ]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
data_dir = "document_collection/"

# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!python ../docparser_v2.py --input-dir {data_dir} --output-dir {data_dir} -c ../docling_v2_config.yaml

In [ ]:
# Step 2: Install Required Dependencies
# Install packages needed for document processing and text chunking

%pip install docling markdown-it-py
%pip install --upgrade transformers

In [ ]:
# Step 3: Load Processed Documents
import glob
import os
from pathlib import Path

# In our example above docling step produces markdown of all the PDF files in document_collection
md_files = sorted(glob.glob(f"{data_dir}/*.md"))

if not md_files:
    raise ValueError(f"No markdown files found in {data_dir}. Run Step 1 first.")

documents = []
for md_path in md_files:
    with open(md_path, "r", encoding="utf-8") as f:
        documents.append(
            {
                "source_file": os.path.basename(md_path),
                "document_outline": Path(md_path).stem,
                "text": f.read(),
            }
        )

print(f"Loaded {len(documents)} markdown files from {data_dir}")
for doc in documents:
    print(f"  - {doc['source_file']}")


In [ ]:
# Step 4: Text Chunking and Dataset Creation

import os
from markdown_it import MarkdownIt
from typing import List
import datasets
import pandas as pd
from dotenv import load_dotenv
from sdg_hub import Flow, FlowRegistry
from sdg_hub.core.utils.error_handling import EmptyDatasetError

load_dotenv()


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    md = MarkdownIt()
    tokens = md.parse(text)

    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                chunks.append(" ".join(current_words))
                current_words = current_words[-overlap:] if overlap > 0 else []

    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


def set_model_config(flow_object):
    model_provider = os.getenv("MODEL_PROVIDER", "hosted_vllm")
    print(f"Using model provider: {model_provider}")

    if model_provider == "hosted_vllm":
        flow_object.set_model_config(
            model=os.getenv("VLLM_MODEL", "hosted_vllm/meta-llama/Llama-3.3-70B-Instruct"),
            api_base=os.getenv("VLLM_API_BASE", "http://localhost:8000/v1"),
            api_key=os.getenv("VLLM_API_KEY", "EMPTY"),
            enable_reasoning=os.getenv("ENABLE_REASONING", "false").lower() in ("1", "true", "yes"),
        )
    elif model_provider == "openai":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
        )
    elif model_provider == "openrouter":
        flow_object.set_model_config(
            model=os.getenv("OPENAI_MODEL", "openai/gpt-4"),
            api_key=os.getenv("OPENAI_API_KEY"),
            api_base="https://openrouter.ai/api/v1",
        )
    elif model_provider == "ollama":
        flow_object.set_model_config(
            model=os.getenv("OLLAMA_MODEL", "ollama/gemma2"),
            api_base=os.getenv("OLLAMA_API_BASE", "http://localhost:11434"),
        )
    elif model_provider == "maas":
        flow_object.set_model_config(
            model=os.getenv("MAAS_MODEL"),
            api_base=os.getenv("MAAS_API_BASE"),
            api_key=os.getenv("MAAS_API_KEY"),
        )
    else:
        raise ValueError(f"Unsupported MODEL_PROVIDER: {model_provider}")

    return flow_object


BASE_ICL_DOCUMENT = (
    "The coastal town of Willow Creek, once renowned for its pristine beaches, now struggles "
    "with rampant pollution. Plastic debris and oil spills have devastated marine life, prompting "
    "a decline in tourism and fishing industries. Residents have organized weekly clean-up "
    "initiatives, but the scale of the problem overwhelms their efforts. Technologists at the "
    "local university have developed an AI-powered buoy system to combat this. The buoys, "
    "equipped with solar panels and filtration technology, can identify and absorb oil spills while "
    "collecting microplastics. Data from the buoys is shared publicly, raising awareness and "
    "pressuring corporations to adopt sustainable practices."
)
BASE_ICL_QUERY_1 = "How does the technological solution address both environmental and economic challenges in the document?"
BASE_ICL_QUERY_2 = "What values are reflected by the community clean-up initiatives and the university buoy project?"
BASE_ICL_QUERY_3 = "If the buoy project succeeds, what possible unintended consequences could emerge?"

all_rows = []
first_chunk_by_file = {}
question_seed_rows = []
domain = os.getenv("SEED_DOMAIN", "general")

for doc in documents:
    chunks = chunk_markdown(doc["text"], max_tokens=5000, overlap=1000)
    if not chunks:
        continue

    source_file = doc["source_file"]
    document_outline = doc["document_outline"]
    first_chunk = " ".join(chunks[0].split()[:250])
    first_chunk_by_file[source_file] = first_chunk

    doc_context = " ".join(doc["text"].split()[:2500])
    question_seed_rows.append(
        {
            "source_file": source_file,
            "document_outline": document_outline,
            "document": doc_context,
            "icl_document": BASE_ICL_DOCUMENT,
            "icl_query_1": BASE_ICL_QUERY_1,
            "icl_query_2": BASE_ICL_QUERY_2,
            "icl_query_3": BASE_ICL_QUERY_3,
            "domain": domain,
        }
    )

    for chunk in chunks:
        all_rows.append(
            {
                "document": chunk,
                "document_outline": document_outline,
                "source_file": source_file,
            }
        )

if not all_rows:
    raise ValueError("No chunks were produced from markdown documents.")

if not question_seed_rows:
    raise ValueError("No documents available for LLM-based ICL query generation.")

FlowRegistry.discover_flows()
flow_name = "Document Based Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
if not flow_path:
    raise ValueError(f"Flow not found: {flow_name}")

flow = Flow.from_yaml(flow_path)
flow = set_model_config(flow)

# Keep only blocks needed to generate and parse question lists.
kept_blocks = []
for block in flow.blocks:
    kept_blocks.append(block)
    if block.block_name == "parse_question_list":
        break
flow.blocks = kept_blocks

prompt_block = next(
    (block for block in flow.blocks if block.block_name == "question_generation_prompt"),
    None,
)
if prompt_block is None:
    raise ValueError("question_generation_prompt block was not found in the flow.")

prompt_path = prompt_block.prompt_config_path.replace("\\", "/")
expected_prompt_suffix = (
    "enhanced_multi_summary_qa/doc_direct_qa/prompts/generate_question_list.yaml"
)
if not prompt_path.endswith(expected_prompt_suffix):
    raise ValueError(
        f"Unexpected question prompt template: {prompt_block.prompt_config_path}. "
        f"Expected suffix: {expected_prompt_suffix}"
    )

print(f"Using question prompt template: {prompt_block.prompt_config_path}")

runtime_params = {
    "question_generation": {
        "max_tokens": int(os.getenv("ICL_QUESTION_MAX_TOKENS", "1024"))
    }
}
max_concurrency = int(os.getenv("MAX_CONCURRENCY", "8"))

question_seed_df = pd.DataFrame(question_seed_rows)

try:
    questions_df = flow.generate(
        question_seed_df,
        runtime_params=runtime_params,
        max_concurrency=max_concurrency,
    )
except EmptyDatasetError as exc:
    if exc.block_name != "parse_question_list":
        raise

    print("parse_question_list returned empty output; retrying with parser fallback.")
    for block in flow.blocks:
        if block.block_name == "parse_question_list":
            block.start_tags = [""]
            block.end_tags = [""]
            break

    questions_df = flow.generate(
        question_seed_df,
        runtime_params=runtime_params,
        max_concurrency=max_concurrency,
    )

if isinstance(questions_df, datasets.Dataset):
    questions_df = questions_df.to_pandas()


def normalize_questions(values):
    result = []
    for value in values:
        if isinstance(value, list):
            result.extend([v.strip() for v in value if isinstance(v, str) and v.strip()])
        elif isinstance(value, str) and value.strip():
            result.append(value.strip())

    unique = []
    seen = set()
    for q in result:
        if q not in seen:
            seen.add(q)
            unique.append(q)
    return unique


queries_by_file = {}
for source_file, group in questions_df.groupby("source_file"):
    questions = normalize_questions(group["question"].tolist())
    if len(questions) >= 3:
        queries_by_file[source_file] = questions[:3]

missing_files = sorted(set(first_chunk_by_file.keys()) - set(queries_by_file.keys()))
if missing_files:
    raise ValueError(
        "Unable to generate at least 3 questions for: " + ", ".join(missing_files)
    )

seed_data = datasets.Dataset.from_list(all_rows)


def add_icl_fields(sample):
    q1, q2, q3 = queries_by_file[sample["source_file"]]
    return {
        "icl_document": first_chunk_by_file[sample["source_file"]],
        "icl_query_1": q1,
        "icl_query_2": q2,
        "icl_query_3": q3,
        "domain": domain,
    }


seed_data = seed_data.map(add_icl_fields)

seed_data_path = os.getenv("SEED_DATA_PATH", "seed_data.jsonl")
seed_data.to_json(seed_data_path, orient="records", lines=True)

print(f"Saved seed data to: {seed_data_path}")
print(f"Total chunks: {len(seed_data)}")
print(f"Columns: {seed_data.column_names}")
print("Example generated queries:")
for source_file, queries in list(queries_by_file.items())[:3]:
    print(f"- {source_file}")
    print(f"  Q1: {queries[0]}")
    print(f"  Q2: {queries[1]}")
    print(f"  Q3: {queries[2]}")


### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook